SETTING THE LLM

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

groq_llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
)

response = groq_llm.invoke("Say 'LLM is working!' and explain RAG in one sentence.")

print(response.content)


LLM is working! RAG (Retrieval‑Augmented Generation) is a technique that blends a language model’s generative output with relevant documents retrieved from an external knowledge base to improve accuracy and contextual relevance.


Setting up the agents

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=groq_llm,
    tools=[],
    checkpointer=memory,      # here memory stores the temp data
    system_prompt="You are a helpful assistant.")

agent.invoke({"message": [{"role": "user", "content": "What is RAG?"}]})

SETTING UP PROMPT

In [5]:
promt = " You are a helpful assistant. You are given a question and you have to answer it."

PREPARING PICKLE FILE

* Getting the data 
* EDA
* preprocessing before split
* split
* preprocesing after split
* model

In [12]:
import pandas as pd
df = pd.read_csv('kyphosis.csv')

#Preprocess before split
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Kyphosis'] = le.fit_transform(df['Kyphosis'])

#Split
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Kyphosis'])
y = df['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#Preprocess after split
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[['Age']] = scaler.fit_transform(X_train[['Age']])
X_test[['Age']] = scaler.transform(X_test[['Age']])

#--------------------------------Model Training--------------------------------
# from sklearn.ensemble import RandomForestClassifier
# dt = RandomForestClassifier()
# dt.fit(X_train, y_train)
# preds = dt.predict(X_test)
# from sklearn.metrics import accuracy_score
# accuracy = accuracy_score(y_test, preds) 
# print(accuracy)

In [13]:

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train,y_train)

preds = dt.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(preds,y_test))

              precision    recall  f1-score   support

           0       0.89      0.77      0.83        22
           1       0.17      0.33      0.22         3

    accuracy                           0.72        25
   macro avg       0.53      0.55      0.53        25
weighted avg       0.81      0.72      0.76        25



# then we dump our pickle file

In [15]:
import pickle

preprocessing = {
    "age_scaler": scaler,
    "label_encoder": le,
}

with open("preprocessing.pkl", "wb") as f:
    pickle.dump(preprocessing, f)

with open("model.pkl", "wb") as f:
    pickle.dump(dt, f)

# testing the pickle file

In [16]:
model = pickle.load(open("model.pkl", "rb"))
preprocessing = pickle.load(open("preprocessing.pkl", "rb"))

patient = pd.DataFrame({"Age": [15], "Number": [12], "Start": [7]})
patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
prediction = model.predict(patient)
prediction

array([1])

#Creating a tol for our agent for utilizing ml pkl file


In [20]:
def predict_kyphosis(age:int, number:int, start:int)-> int:
    """ This function takes age, number, and start as input and returns the predicted kyphosis class (0 or 1) using the trained model. """
    
    model = pickle.load(open("model.pkl", "rb"))
    preprocessing = pickle.load(open("preprocessing.pkl", "rb"))
    patient = pd.DataFrame({"Age": [age], "Number": [number], "Start": [start]})
    patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
    prediction = model.predict(patient)
    return preprocessing["label_encoder"].inverse_transform(prediction)[0]

# intregating  out kyphosis model into our agent

In [25]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

#memory = InMemorySaver()

agent = create_agent(
    model=groq_llm,
    tools=[predict_kyphosis],
   # checkpointer=memory,      # here memory stores the temp data
    system_prompt="""
    You are a helpful assistant.

    Whenever the user provides a child's spinal surgery information
    and asks about kyphosis, ALWAYS call the predict_kyphosis tool.
    Do not make predictions yourself.

    Use the tool's output to explain the prediction.
    Clarify that this is a machine-learning prediction, not a medical
    diagnosis.
    """)
agent.invoke({"messages": [{"role": "user", "content": "A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?"}]})

{'messages': [HumanMessage(content='A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?', additional_kwargs={}, response_metadata={}, id='84da020f-0f58-4f8b-97cb-9edcc24d058a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user is asking about a child aged 71 months, 3 vertebrae involved, starting at vertebra 5. The user wants to know if they will get kyphosis. According to the instructions, we must ALWAYS call the predict_kyphosis tool with age, number, start. Then use the tool's output to explain the prediction, clarify that it's a machine-learning prediction, not a medical diagnosis.\n\nWe should not give any direct answer before calling the tool. So we call the function with age=71, number=3, start=5.", 'tool_calls': [{'id': 'fc_6582602d-4bbc-462e-aa60-3f7e5ae8ec32', 'function': {'arguments': '{"age":71,"number":3,"start":5}', 'name': 'predict_kyphosis'}, 'type': 'function'}]}, response_metadata={'token_usage': {'com

In [ ]:
patient = pd.DataFrame({"Age": [71], "Number": [3], "Start": [5]})
patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
prediction = model.predict(patient)
return preprocessing["label_encoder"].inverse_transform(prediction)[0]